In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
train = pd.read_csv('train.csv').drop('id', axis=1)
test = pd.read_csv('test.csv').drop('id', axis=1)
sample_submission = pd.read_csv('sample_submission.csv')

In [3]:
X, y = train.drop('Churn', axis=1), train['Churn']
y = y.map({'Yes': 1, 'No': 0})

In [4]:
X.head(3)

,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Male,0,Yes,Yes,29,Yes,No,DSL,Yes,No,Yes,Yes,No,No,One year,Yes,Mailed check,60.1,1653.85
1,Male,0,Yes,Yes,58,Yes,No,DSL,Yes,Yes,No,Yes,Yes,No,Two year,No,Credit card (automatic),69.5,3778.20
2,Male,0,Yes,No,58,Yes,Yes,Fiber optic,No,Yes,No,No,Yes,Yes,Month-to-month,Yes,Electronic check,100.4,5841.35


In [10]:
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

cat_features = X.select_dtypes(include='object').columns.tolist()
cat_features.append('SeniorCitizen')

oof_preds = np.zeros(len(X))
test_preds = np.zeros(len(test))

for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y)):

    print(f'Fold {fold+1}')

    X_train, X_valid = X.iloc[train_idx], X.iloc[valid_idx]
    y_train, y_valid = y.iloc[train_idx], y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=2000,
        learning_rate=0.05,
        depth=6,
        eval_metric='AUC',
        random_seed=42,
        verbose=200,
        early_stopping_rounds=200
    )

    model.fit(
        X_train, y_train,
        cat_features=cat_features,
        eval_set=(X_valid, y_valid),
        use_best_model=True
    )

    oof_preds[valid_idx] = model.predict_proba(X_valid)[:, 1]
    test_preds += model.predict_proba(test)[:, 1] / skf.n_splits

Fold 1
0:	test: 0.8723458	best: 0.8723458 (0)	total: 323ms	remaining: 10m 44s
200:	test: 0.9134595	best: 0.9134595 (200)	total: 32s	remaining: 4m 46s
400:	test: 0.9147394	best: 0.9147394 (400)	total: 1m 8s	remaining: 4m 34s
600:	test: 0.9153307	best: 0.9153307 (600)	total: 1m 45s	remaining: 4m 6s
800:	test: 0.9156484	best: 0.9156484 (800)	total: 2m 21s	remaining: 3m 31s
1000:	test: 0.9158147	best: 0.9158157 (998)	total: 2m 54s	remaining: 2m 54s
1200:	test: 0.9159101	best: 0.9159105 (1199)	total: 3m 30s	remaining: 2m 20s
1400:	test: 0.9159954	best: 0.9159959 (1396)	total: 4m 4s	remaining: 1m 44s
1600:	test: 0.9160554	best: 0.9160560 (1597)	total: 4m 38s	remaining: 1m 9s
1800:	test: 0.9160738	best: 0.9160761 (1794)	total: 5m 11s	remaining: 34.4s
1999:	test: 0.9160579	best: 0.9160761 (1794)	total: 5m 45s	remaining: 0us

bestTest = 0.9160761386
bestIteration = 1794

Shrink model to first 1795 iterations.
Fold 2
0:	test: 0.8743336	best: 0.8743336 (0)	total: 228ms	remaining: 7m 36s
200:	test

In [11]:
from sklearn.metrics import roc_auc_score

cv_score = roc_auc_score(y, oof_preds)
print("CV ROC-AUC:", cv_score)

CV ROC-AUC: 0.9164323191646635


In [12]:
submission = sample_submission.copy()
submission['Churn'] = test_preds
submission.to_csv('submission.csv', index=False)